In [85]:
import os
import json
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader, random_split
from PIL import Image
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [86]:
# ==========================================
# 1. CARICAMENTO DATABASE JSON
# ==========================================
def load_database(filepath="database.json"):
    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)
    return {item["id"]: item for item in data}

In [87]:
# ==========================================
# 2. DATA AUGMENTATION & PIPELINE TRASFORMAZIONI
# ==========================================
train_transform_pipeline = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform_pipeline = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 3. Caricamento Dati
db = load_database("database.json")
full_dataset = datasets.ImageFolder(root="dataset_image", transform=train_transform_pipeline)

# Split 80% Train, 20% Validation
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False)

print(f"✓ Dataset caricato: {len(full_dataset)} immagini totali suddivise in {train_size} train e {val_size} validation.")

✓ Dataset caricato: 30 immagini totali suddivise in 24 train e 6 validation.


In [88]:
# ==========================================
# 3. CLASSE EARLY STOPPING
# ==========================================
class EarlyStopping:
    def __init__(self, patience=2, min_delta=0.001, checkpoint_path="best_chefbot_model.pt"):
        self.patience = patience
        self.min_delta = min_delta
        self.checkpoint_path = checkpoint_path
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss, model):
        if self.best_loss is None:
            self.best_loss = val_loss
            self.save_checkpoint(model)
        elif val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.save_checkpoint(model)
            self.counter = 0
        else:
            self.counter += 1
            print(f"➜ EarlyStopping: {self.counter}/{self.patience} epoche senza miglioramento.")
            if self.counter >= self.patience:
                self.early_stop = True

    def save_checkpoint(self, model):
        torch.save(model.state_dict(), self.checkpoint_path)
        print(f"  [✓] Nuovo miglioramento! Modello salvato in '{self.checkpoint_path}'")

def build_food_model(num_classes=15):
    model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
    for param in model.parameters():
        param.requires_grad = False
        
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, num_classes)
    )
    return model

In [89]:
# ==========================================
# 4. COMPUTER VISION (MobileNetV2 Transfer Learning)
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = build_food_model(num_classes=len(db)).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.0003)
early_stopper = EarlyStopping(patience=2, checkpoint_path="best_chefbot_model.pt")

num_epochs = 10
print("--- INIZIO ADDESTRAMENTO ---")
for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * images.size(0)
        
    train_loss /= len(train_loader.dataset)
    
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            
    val_loss /= len(val_loader.dataset)
    
    print(f"Epoca [{epoch+1:02d}/{num_epochs:02d}] - Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    
    early_stopper(val_loss, model)
    if early_stopper.early_stop:
        print(f"\n[!] Early Stopping attivato all'epoca {epoch+1}.")
        break

# Caricamento dei pesi salvati nel file best_chefbot_model.pt
model.load_state_dict(torch.load("best_chefbot_model.pt"))
print("✓ Pesi migliori caricati da 'best_chefbot_model.pt'.")

--- INIZIO ADDESTRAMENTO ---
Epoca [01/10] - Train Loss: 2.7600 | Val Loss: 2.7876
  [✓] Nuovo miglioramento! Modello salvato in 'best_chefbot_model.pt'
Epoca [02/10] - Train Loss: 2.6585 | Val Loss: 2.7833
  [✓] Nuovo miglioramento! Modello salvato in 'best_chefbot_model.pt'
Epoca [03/10] - Train Loss: 2.6177 | Val Loss: 2.8085
➜ EarlyStopping: 1/2 epoche senza miglioramento.
Epoca [04/10] - Train Loss: 2.5765 | Val Loss: 2.8541
➜ EarlyStopping: 2/2 epoche senza miglioramento.

[!] Early Stopping attivato all'epoca 4.
✓ Pesi migliori caricati da 'best_chefbot_model.pt'.


In [90]:
# ==========================================
# CELLA 5: LOOKUP METADATI E RICERCA SEMANTICA (TOP 3)
# ==========================================

# Function per recuperare la scheda nutrizionale
def lookup_food_by_id(db, class_id):
    return db.get(class_id, {"error": "Alimento non trovato"})

# Classe per la Ricerca Semantica con Embeddings
class MoodSearchEngine:
    def __init__(self, db):
        self.db = db
        self.encoder = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
        self.food_embeddings = []
        self.ids = []
        self._build_embeddings()

    def _build_embeddings(self):
        texts_to_embed = []
        for food_id, item in self.db.items():
            self.ids.append(food_id)
            combined_text = f"{item['name']}: {item['description']}"
            texts_to_embed.append(combined_text)
            
        self.food_embeddings = self.encoder.encode(texts_to_embed, normalize_embeddings=True)

    def search_by_mood(self, user_mood_query, top_k=3):
        """Restituisce i top_k piatti più simili alla query dell'utente."""
        query_embedding = self.encoder.encode([user_mood_query], normalize_embeddings=True)
        similarities = cosine_similarity(query_embedding, self.food_embeddings)[0]
        top_indices = np.argsort(similarities)[::-1][:top_k]
        
        results = []
        for idx in top_indices:
            food_id = self.ids[idx]
            results.append({
                "food": self.db[food_id],
                "similarity_score": float(similarities[idx])
            })
        return results

# --- TEST RICERCA SEMANTICA CON I 3 MIGLIORI RISULTATI ---
search_engine = MoodSearchEngine(db)
query = "voglio qualcosa di fresco e leggero"
risultati = search_engine.search_by_mood(query, top_k=3)

print("--- TEST RICERCA SEMANTICA (TOP 3 RISULTATI) ---")
print(f"Query Utente: \"{query}\"\n")

for i, res in enumerate(risultati, 1):
    food = res['food']
    print(f"#{i} Consigliato: {food['name']} (Score: {res['similarity_score']:.4f})")
    print(f"   Descrizione: {food['description']}")
    print(f"   Valori Nutrizionali: {food['nutritional_values']}")
    print(f"   Ingredienti: {', '.join(food['ingredients'])}\n")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2310.60it/s]


--- TEST RICERCA SEMANTICA (TOP 3 RISULTATI) ---
Query Utente: "voglio qualcosa di fresco e leggero"

#1 Consigliato: Insalata Caprese (Score: 0.4790)
   Descrizione: Piatto estivo crudo, esteticamente fresco, molto leggero, poco calorico, digeribile e salutare.
   Valori Nutrizionali: {'calories': 177, 'protein_g': 10, 'carbs_g': 3, 'fat_g': 14}
   Ingredienti: pomodori, mozzarella di bufala, basilico, olio d'oliva, sale

#2 Consigliato: Gelato al Cioccolato (Score: 0.4789)
   Descrizione: Dessert freddo, cremoso e dolcissimo a base di cacao e panna.
   Valori Nutrizionali: {'calories': 216, 'protein_g': 4, 'carbs_g': 28, 'fat_g': 10}
   Ingredienti: latte, panna, zucchero, cacao in polvere, cioccolato fondente

#3 Consigliato: Croissant (Score: 0.4355)
   Descrizione: Prodotto da forno sfogliato, burroso e fragrante per la colazione.
   Valori Nutrizionali: {'calories': 406, 'protein_g': 8, 'carbs_g': 45, 'fat_g': 21}
   Ingredienti: farina, burro, zucchero, lievito di birra, latte



In [91]:
# ==========================================
# CELLA 6: PIPELINE UNIFICATA CHEFBOT AI (VISION + NLP)
# ==========================================

def predict_image(model, image_path):
    """
    Classifica un'immagine singola caricata da disco usando il modello MobileNetV2.
    """
    model.eval()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    
    # Caricamento e trasformazione dell'immagine
    image = Image.open(image_path).convert('RGB')
    tensor_img = val_transform_pipeline(image).unsqueeze(0).to(device)
    
    with torch.no_grad():
        outputs = model(tensor_img)
        _, predicted_idx = torch.max(outputs, 1)
        
    return predicted_idx.item()


def chefbot_pipeline(image_path, user_mood_query, model, db, search_engine, top_k=3):
    """
    Orchestratore principale: 
    1. Riconosce il piatto dalla foto caricata via Computer Vision.
    2. Mostra la scheda nutrizionale e i metadati.
    3. Offre consigli alternativi basati sull'umore/desiderio espresso via testo dall'utente.
    """
    print("="*60)
    print("                CHEFBOT AI - ASSISTENTE INTELLIGENTE")
    print("="*60)
    
    # 1. ANALISI VISIVA DALL'IMMAGINE
    predicted_id = predict_image(model, image_path)
    identified_food = lookup_food_by_id(db, predicted_id)
    
    print("\n--- 1. RICONOSCIMENTO VISIVO (Computer Vision) ---")
    print(f"📷 Immagine analizzata: {image_path}")
    print(f"🍽️  Piatto Riconosciuto: {identified_food['name']}")
    print(f"📊 Valori Nutrizionali: {identified_food['nutritional_values']}")
    print(f"📝 Descrizione: {identified_food['description']}")
    
    # 2. RICERCA SEMANTICA PER UMORE
    print("\n--- 2. CONSIGLI PERSONALIZZATI PER UMORE (NLP) ---")
    print(f"💬 Richiesta Utente: \"{user_mood_query}\"")
    
    mood_recommendations = search_engine.search_by_mood(user_mood_query, top_k=top_k)
    
    print(f"\nTop {top_k} piatti consigliati per la tua richiesta:")
    for i, res in enumerate(mood_recommendations, 1):
        food = res['food']
        print(f"  #{i} {food['name']} (Affinità: {res['similarity_score']:.4f})")
        print(f"     ➔ {food['description']}")
        print(f"     ➔ Calorie: {food['nutritional_values']['calories']} kcal | Proteine: {food['nutritional_values']['protein_g']}g")
    
    print("="*60)


# ==========================================
# ESEMPIO DI TEST COMPLETO DELLA PIPELINE
# ==========================================

# Scegliamo un'immagine di test presente nella cartella dataset_image
test_image_path = "dataset_image/caprese_salad/caprese_salad_1.jpg"
test_query = "voglio qualcosa di fresco e leggero"

# Esecuzione dell'assistente completo
chefbot_pipeline(
    image_path=test_image_path,
    user_mood_query=test_query,
    model=model,
    db=db,
    search_engine=search_engine,
    top_k=3
)

                CHEFBOT AI - ASSISTENTE INTELLIGENTE

--- 1. RICONOSCIMENTO VISIVO (Computer Vision) ---
📷 Immagine analizzata: dataset_image/caprese_salad/caprese_salad_1.jpg
🍽️  Piatto Riconosciuto: Insalata Caprese
📊 Valori Nutrizionali: {'calories': 177, 'protein_g': 10, 'carbs_g': 3, 'fat_g': 14}
📝 Descrizione: Piatto estivo crudo, esteticamente fresco, molto leggero, poco calorico, digeribile e salutare.

--- 2. CONSIGLI PERSONALIZZATI PER UMORE (NLP) ---
💬 Richiesta Utente: "voglio qualcosa di fresco e leggero"

Top 3 piatti consigliati per la tua richiesta:
  #1 Insalata Caprese (Affinità: 0.4790)
     ➔ Piatto estivo crudo, esteticamente fresco, molto leggero, poco calorico, digeribile e salutare.
     ➔ Calorie: 177 kcal | Proteine: 10g
  #2 Gelato al Cioccolato (Affinità: 0.4789)
     ➔ Dessert freddo, cremoso e dolcissimo a base di cacao e panna.
     ➔ Calorie: 216 kcal | Proteine: 4g
  #3 Croissant (Affinità: 0.4355)
     ➔ Prodotto da forno sfogliato, burroso e fragrante